# Reasoning vs Non-Reasoning Models

A reasoning model is not a normal model with a better prompt. It is a model trained to
spend tokens thinking before it answers, and you are billed for that thinking whether or
not you ever see it.

This notebook makes the difference measurable rather than anecdotal. You will run the
**same task** through a reasoning model and a non-reasoning one and compare four numbers
that actually decide which to use: **latency, output tokens, reasoning tokens, and
whether the answer was right**.

The part most write-ups skip, and the part this notebook insists on: **a task where the
reasoning model is the wrong choice.** If you only ever benchmark on puzzles, you learn
"always use the smart one", which is expensive and wrong.

## Learning objectives

By the end you can:

1. Explain what a reasoning model does differently, in terms of tokens rather than vibes.
2. Measure latency, token spend and correctness for two models on one task.
3. Show a case where reasoning pays for itself, and a case where it burns money for nothing.
4. Read the `usage` object and find the tokens you were billed for but never received.
5. State a defensible rule for when to reach for a reasoning model.

## Where this fits

- **Stage:** `01_Foundations/` — no framework needed, so no LangChain and no LangGraph here.
- **Next:** `02_Reasoning_Effort_Levers.ipynb` turns effort into a dial and finds the knee.
- **Applied:** routing between tiers at runtime is a framework question and lives in
  `02_Core/05_AI_Agent_Fundamentals/4. Workflow_Pattern/2. Routing/`.

## Prerequisites

`OPENAI_API_KEY` in the project-root `.env`, and an account that can reach both models named
below. Running every cell costs a few cents.

## Why this notebook uses the raw `openai` SDK

Everywhere else in this repo you would write `from helpers import get_llm`. Not here, and
not by accident — `01_Foundations/CLAUDE.md` forbids it for this whole stage:

> Neither phase uses the `helpers` factory, and that is deliberate. This stage teaches raw
> provider SDKs [...] wrapping them in `get_llm()` would hide the subject.

That applies with full force here. The subject **is** the provider parameter and the
provider's `usage` object. A factory whose job is to hide which provider you are on would
hide exactly what we are trying to look at.

That said, the factory is not the obstacle it once was. `get_llm()` gained a
`reasoning_effort` passthrough, and its Databricks path returns a `ChatOpenAI` pointed at the
workspace AI Gateway — so the field is present there too. The reason this notebook still uses
the raw client is the stage rule and the subject, not a capability gap.

In [ ]:
# ============ SETUP ============
import json
import os
import time
from dataclasses import dataclass, field

from dotenv import load_dotenv
from openai import OpenAI

load_dotenv()  # expects OPENAI_API_KEY in the project-root .env
client = OpenAI()

# Both names are already used elsewhere in this repo, so you are not juggling two sets.
# Model availability changes often — check what your account can actually reach.
REASONING_MODEL = "o4-mini"      # thinks before answering, bills you for the thinking
FAST_MODEL      = "gpt-4o-mini"  # answers directly

# Published prices move. Edit these to today's numbers rather than trusting a constant
# someone committed months ago. USD per 1M tokens, (input, output).
PRICING = {
    REASONING_MODEL: (1.10, 4.40),
    FAST_MODEL:      (0.15, 0.60),
}

print(f"reasoning : {REASONING_MODEL}")
print(f"fast      : {FAST_MODEL}")

## 1. The difference is in the model, not the prompt

"Think step by step" is a *prompt* technique: you ask a normal model to emit its
reasoning as part of the answer, and you can read every word of it.

A reasoning model is a *training* difference. It produces reasoning tokens internally,
decides on its own how many to spend, and returns only the conclusion. You are charged
for those tokens at the output rate, and in most cases you never see them.

Three consequences that matter in production, all of which we will measure:

| | Non-reasoning | Reasoning |
|---|---|---|
| Who decides how much thinking happens | You, via the prompt | The model |
| Can you read the reasoning | Yes, it is in the response | Usually no |
| What you are billed for | What you can see | What you can see **plus** what you cannot |

In [ ]:
# ============ MEASUREMENT HARNESS ============
@dataclass
class Run:
    """One model call, with the numbers that decide whether it was worth it."""
    model: str
    label: str
    text: str
    seconds: float
    prompt_tokens: int
    completion_tokens: int
    reasoning_tokens: int = 0
    correct: bool | None = field(default=None)

    @property
    def cost_usd(self) -> float:
        inp, out = PRICING.get(self.model, (0.0, 0.0))
        # reasoning tokens are already inside completion_tokens; they bill at the output rate
        return (self.prompt_tokens / 1e6) * inp + (self.completion_tokens / 1e6) * out


def ask(model: str, prompt: str, label: str = "") -> Run:
    """Call a model once and capture timing plus the full usage breakdown."""
    started = time.perf_counter()
    resp = client.chat.completions.create(
        model=model,
        messages=[{"role": "user", "content": prompt}],
    )
    elapsed = time.perf_counter() - started

    usage = resp.usage
    # Only reasoning models populate this; .get-style access keeps the call site uniform.
    details = getattr(usage, "completion_tokens_details", None)
    reasoning = getattr(details, "reasoning_tokens", 0) or 0

    return Run(
        model=model,
        label=label,
        text=(resp.choices[0].message.content or "").strip(),
        seconds=elapsed,
        prompt_tokens=usage.prompt_tokens,
        completion_tokens=usage.completion_tokens,
        reasoning_tokens=reasoning,
    )


def compare(prompt: str, label: str) -> list[Run]:
    """Run the identical prompt through both models."""
    return [ask(FAST_MODEL, prompt, label), ask(REASONING_MODEL, prompt, label)]


def show(runs: list[Run]) -> None:
    head = f"{'model':<16}{'sec':>7}{'out tok':>9}{'reasoning':>11}{'$':>10}  answer"
    print(head)
    print("-" * (len(head) + 20))
    for r in runs:
        answer = r.text.replace("\n", " ")[:46]
        print(f"{r.model:<16}{r.seconds:>7.2f}{r.completion_tokens:>9}"
              f"{r.reasoning_tokens:>11}{r.cost_usd:>10.5f}  {answer}")

## 2. A task where reasoning earns its cost

The task below has a property that matters: **you cannot get it right by pattern-matching.**
It needs several constrained steps held together, and a wrong step early produces a
confidently wrong answer rather than an obviously broken one.

That is the shape of problem reasoning models are for.

In [ ]:
# ============ TASK A: MULTI-STEP CONSTRAINT ============
TASK_A = """Five services deploy in a fixed order, one per day, Monday to Friday.

- auth deploys at some point before billing.
- search deploys the day immediately after billing.
- notify deploys on Monday.
- reporting does not deploy on Friday.

Which service deploys on Friday? Reply with the service name only."""

TASK_A_ANSWER = "search"

runs_a = compare(TASK_A, "constraint")
for r in runs_a:
    r.correct = TASK_A_ANSWER in r.text.lower()
show(runs_a)

### Discussion of the output

Look at the **reasoning** column. The fast model shows `0` — every token it produced is in
the answer you can read. The reasoning model shows a number that is often several hundred,
none of which appears in `answer`.

Then look at cost and seconds against `correct`. If the reasoning model is right and the
fast one is wrong, a 5–20× cost multiple on a cheap call is trivially worth it. **Getting a
scheduling answer wrong is not made cheaper by getting it wrong quickly.**

Re-run the cell a couple of times. The fast model on a task like this is frequently not
*consistently* wrong — it is *sometimes* right, which is worse, because it means a single
happy-path test would have told you the cheap model was fine.

## 3. A task where reasoning is waste

Now the case that keeps the rule honest. The task below is unambiguous, single-step, and
mechanical. There is nothing to reason about — the answer is in the input.

In [ ]:
# ============ TASK B: MECHANICAL EXTRACTION ============
TASK_B = """Extract the order id and the total from this line. Reply as JSON with keys
"order_id" and "total", nothing else.

Order #A-4417 shipped 2026-03-02, 3 items, total $148.20, customer 88213."""


def parsed_ok(text: str) -> bool:
    cleaned = text.strip().removeprefix("```json").removeprefix("```").removesuffix("```")
    try:
        obj = json.loads(cleaned)
    except json.JSONDecodeError:
        return False
    return str(obj.get("order_id", "")).endswith("A-4417") and "148.2" in str(obj.get("total", ""))


runs_b = compare(TASK_B, "extraction")
for r in runs_b:
    r.correct = parsed_ok(r.text)
show(runs_b)

### Discussion of the output

Both models should be correct. Now the numbers are the whole argument:

- The reasoning model is **slower**, usually by a multiple, not a margin.
- It **costs more**, partly for reasoning tokens spent deciding that there was nothing to decide.
- The answer is **no better**. It cannot be — the task has one right output.

This is the case that makes "always use the strongest model" an expensive habit. On a
high-volume extraction endpoint, that difference is the entire infrastructure bill, paid
for zero quality.

Notice also that the reasoning model still burned reasoning tokens here. It does not know
in advance that the task is trivial. **Deciding not to think is itself something it pays to
do** — which is the argument for routing, rather than for one model everywhere.

## 4. The invisible bill

`completion_tokens` is the number you are charged for. `reasoning_tokens` is the slice of it
you never received. The gap is the point.

If you estimate cost from the length of the text you got back — a very common shortcut —
you will under-count reasoning-model spend, sometimes by a lot.

In [ ]:
# ============ WHERE THE OUTPUT TOKENS WENT ============
print(f"{'task':<12}{'model':<16}{'billed out':>12}{'visible':>9}{'unseen':>8}{'unseen %':>10}")
print("-" * 67)
for r in runs_a + runs_b:
    visible = r.completion_tokens - r.reasoning_tokens
    pct = (r.reasoning_tokens / r.completion_tokens * 100) if r.completion_tokens else 0
    print(f"{r.label:<12}{r.model:<16}{r.completion_tokens:>12}{visible:>9}"
          f"{r.reasoning_tokens:>8}{pct:>9.0f}%")

## 5. The decision, as a table

Everything above collapses into one question: **does this task have a step where being
wrong is expensive and pattern-matching is not enough?**

In [ ]:
# ============ SUMMARY ACROSS BOTH TASKS ============
print(f"{'task':<12}{'model':<16}{'correct':>9}{'sec':>7}{'$':>10}")
print("-" * 54)
for r in runs_a + runs_b:
    print(f"{r.label:<12}{r.model:<16}{str(r.correct):>9}{r.seconds:>7.2f}{r.cost_usd:>10.5f}")

fast_a, reas_a = runs_a
fast_b, reas_b = runs_b
print(f"\nconstraint task : reasoning cost {reas_a.cost_usd / max(fast_a.cost_usd, 1e-9):.1f}x "
      f"and took {reas_a.seconds / max(fast_a.seconds, 1e-9):.1f}x longer")
print(f"extraction task : reasoning cost {reas_b.cost_usd / max(fast_b.cost_usd, 1e-9):.1f}x "
      f"and took {reas_b.seconds / max(fast_b.seconds, 1e-9):.1f}x longer, for the same answer")

## Key takeaways

1. **A reasoning model is a training difference, not a prompt.** It chooses how long to
   think, and returns a conclusion rather than its working.
2. **You are billed for tokens you never see.** `completion_tokens` includes
   `reasoning_tokens`; estimating cost from visible output under-counts.
3. **Reach for reasoning when a wrong intermediate step is expensive** — multi-constraint
   decisions, planning, anything where a confident wrong answer costs more than latency does.
4. **Do not reach for it on mechanical work.** On unambiguous extraction it is slower,
   dearer, and no more correct. It pays to think about a task with nothing to think about.
5. **Benchmark on your own hard case *and* your own easy case.** Testing only the hard one
   is how teams end up paying reasoning prices for extraction traffic.
6. **Check that the parameter actually reached the model.** If reasoning tokens stay at zero
   while you raise effort, it did not — and you will conclude "effort makes no difference"
   from a measurement that never ran.

### Next

- `02_Reasoning_Effort_Levers.ipynb` — how much thinking, not just whether.
- `02_Core/05_AI_Agent_Fundamentals/4. Workflow_Pattern/2. Routing/` — letting a cheap model
  decide, per request, which tier should handle the work.